# La Plata: elecciones legislativas, 2013-2025

Mismo distrito que los notebooks anteriores (La Plata, Provincia de Buenos Aires, Generales), ahora para las elecciones legislativas: **2013, 2017, 2021 y 2025**. Tres niveles:

- **nacional**: Diputados Nacionales + Senadores Nacionales
- **provincial**: Diputados Provinciales + Senadores Provinciales
- **municipal**: Concejales

El `idCargo` de cada cargo se resolvió igual que en el notebook anterior: probando valores contra `resultado/totalizadocsv` y leyendo `cargo_nombre`. Para La Plata, estable en los cuatro años:

| idCargo | cargo | nivel |
|---|---|---|
| 2 | Senador Nacional | nacional |
| 3 | Diputado(s) Nacional(es) | nacional |
| 5 | Senadores Provinciales | provincial |
| 6 | Diputado(s) Provincial(es) | provincial |
| 10 | Concejales | municipal |

**Limitación real encontrada en 2025**: en la Provincia de Buenos Aires las elecciones legislativas provinciales y municipales de 2025 se desdoblaron de la nacional (se votaron en fecha distinta, administradas por la Junta Electoral provincial, no por la nacional). Este sistema (del Ministerio del Interior, nacional) **solo tiene Diputados Nacionales para La Plata/2025** — ni Senadores, ni Diputados Provinciales, ni Concejales están disponibles acá para ese año. Se confirmó barriendo `idCargo` 1-25 sin encontrar nada más.

El caché de Generales queda en `data/<año>/<nivel>/generales/` (hermano de `paso/`, agregado en la sección 6-7 de este notebook — §2.3 del plan de correcciones).

In [1]:
import io
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from electoral.client import ResultadosClient, ResultadosNoDisponibles
from electoral.models import ResultadoElectoral

REPO = Path.cwd().parent
client = ResultadosClient(cache_dir=REPO / "data")

ANIOS = [2013, 2017, 2021, 2025]

# (nivel, categoria_id, nombre) — nivel decide la carpeta de caché (data/<anio>/<nivel>/);
# nombre es solo para identificar el cargo en los prints de este notebook.
CARGOS = [
    ("nacional", 2, "senador_nacional"),
    ("nacional", 3, "diputados_nacionales"),
    ("provincial", 5, "senadores_provinciales"),
    ("provincial", 6, "diputados_provinciales"),
    ("municipal", 10, "concejales"),
]

LA_PLATA = dict(
    tipo_eleccion=2,  # Generales
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

## 1. Traer el CSV oficial de cada (año, cargo) disponible

In [2]:
dataframes = {}
faltantes = []
for anio in ANIOS:
    for nivel, categoria_id, nombre in CARGOS:
        try:
            csv_bytes = client.get_resultados_csv(
                anio_eleccion=anio, categoria_nombre=f"{nivel}/generales", categoria_id=categoria_id, **LA_PLATA
            )
        except ResultadosNoDisponibles:
            faltantes.append((anio, nombre))
            continue
        df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
        df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
        dataframes[(anio, categoria_id)] = {"nivel": nivel, "nombre": nombre, "df": df}
        print(f"{anio}/{nivel}/{nombre} (idCargo={categoria_id}): {len(df)} filas, "
              f"{df['mesa_id'].nunique()} mesas, cargo_nombre={df['cargo_nombre'].unique()}")

print("\nno disponibles (esperado, no todo se vota todos los años):")
for anio, nombre in faltantes:
    print(f"  {anio}/{nombre}")

2013/nacional/diputados_nacionales (idCargo=3): 14980 filas, 1524 mesas, cargo_nombre=<StringArray>
['DIPUTADO NACIONAL']
Length: 1, dtype: str
2013/provincial/diputados_provinciales (idCargo=6): 15240 filas, 1524 mesas, cargo_nombre=<StringArray>
['DIPUTADO PROVINCIAL']
Length: 1, dtype: str
2013/municipal/concejales (idCargo=10): 19812 filas, 1524 mesas, cargo_nombre=<StringArray>
['CONCEJALES/AS']
Length: 1, dtype: str
2017/nacional/senador_nacional (idCargo=2): 16630 filas, 1663 mesas, cargo_nombre=<StringArray>
['SENADOR NACIONAL']
Length: 1, dtype: str
2017/nacional/diputados_nacionales (idCargo=3): 16630 filas, 1663 mesas, cargo_nombre=<StringArray>
['DIPUTADO NACIONAL']
Length: 1, dtype: str


2017/provincial/diputados_provinciales (idCargo=6): 19956 filas, 1663 mesas, cargo_nombre=<StringArray>
['DIPUTADO PROVINCIAL']
Length: 1, dtype: str
2017/municipal/concejales (idCargo=10): 18150 filas, 1650 mesas, cargo_nombre=<StringArray>
['CONCEJALES']
Length: 1, dtype: str
2021/nacional/diputados_nacionales (idCargo=3): 17281 filas, 1571 mesas, cargo_nombre=<StringArray>
['DIPUTADOS NACIONALES']
Length: 1, dtype: str


2021/provincial/diputados_provinciales (idCargo=6): 16720 filas, 1672 mesas, cargo_nombre=<StringArray>
['DIPUTADOS PROVINCIALES']
Length: 1, dtype: str
2021/municipal/concejales (idCargo=10): 16720 filas, 1672 mesas, cargo_nombre=<StringArray>
['CONCEJALES']
Length: 1, dtype: str


2025/nacional/diputados_nacionales (idCargo=3): 33580 filas, 1679 mesas, cargo_nombre=<StringArray>
['DIPUTADO NACIONAL']
Length: 1, dtype: str



no disponibles (esperado, no todo se vota todos los años):
  2013/senador_nacional
  2013/senadores_provinciales
  2017/senadores_provinciales
  2021/senador_nacional
  2021/senadores_provinciales
  2025/senador_nacional
  2025/senadores_provinciales
  2025/diputados_provinciales
  2025/concejales


## 2. Resultado de cada (año, cargo) disponible

In [3]:
for (anio, categoria_id), info in dataframes.items():
    df = info["df"]
    positivos = (
        df[df["votos_tipo"] == "POSITIVO"]
        .groupby("agrupacion_nombre")["votos_cantidad"]
        .sum()
        .sort_values(ascending=False)
    )
    print(f"=== {anio}/{info['nivel']}/{info['nombre']} ===")
    print(positivos.head(5).to_string())
    print()

=== 2013/nacional/diputados_nacionales ===
agrupacion_nombre
FRENTE RENOVADOR                             140382
FRENTE PARA LA VICTORIA                      103224
FRENTE PROGRESISTA CIVICO Y SOCIAL            70580
UNIDOS POR LA LIBERTAD Y EL TRABAJO           31132
FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES     30672

=== 2013/provincial/diputados_provinciales ===
agrupacion_nombre
FRENTE RENOVADOR                             133030
FRENTE PARA LA VICTORIA                      108787
FRENTE PROGRESISTA CIVICO Y SOCIAL            70473
UNIDOS POR LA LIBERTAD Y EL TRABAJO           32985
FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES     32480

=== 2013/municipal/concejales ===
agrupacion_nombre
FRENTE RENOVADOR                                 123696
FRENTE PARA LA VICTORIA                           68432
FRENTE PROGRESISTA CIVICO Y SOCIAL                64167
FRENTE SOCIAL DE LA PROVINCIA DE BUENOS AIRES     43100
UNIDOS POR LA LIBERTAD Y EL TRABAJO               29363

=== 2017/nacional/s

## 3. Validar contra el agregado de la API (JSON)


In [4]:
def normalizar_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


for (anio, categoria_id), info in dataframes.items():
    df = info["df"]
    positivos_csv = df[df["votos_tipo"] == "POSITIVO"].groupby("agrupacion_id")["votos_cantidad"].sum()
    positivos_csv.index = positivos_csv.index.map(normalizar_id)

    raw = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=f"{info['nivel']}/generales", categoria_id=categoria_id, **LA_PLATA
    )
    resultado = ResultadoElectoral.from_json(raw)
    positivos_api = {
        normalizar_id(a.id_agrupacion): a.votos for a in resultado.valores_totalizados_positivos
    }

    agrupaciones = sorted(set(positivos_csv.index) | set(positivos_api))
    diffs = [ag for ag in agrupaciones if positivos_csv.get(ag, 0) != positivos_api.get(ag, 0)]
    estado = "OK" if not diffs else f"AGREGADO JSON NO CONFIABLE (difiere en {len(diffs)} agrupaciones)"
    print(f"{anio}/{info['nivel']}/{info['nombre']}: {estado}")

2013/nacional/diputados_nacionales: OK
2013/provincial/diputados_provinciales: OK
2013/municipal/concejales: OK
2017/nacional/senador_nacional: OK
2017/nacional/diputados_nacionales: OK
2017/provincial/diputados_provinciales: OK
2017/municipal/concejales: OK
2021/nacional/diputados_nacionales: OK
2021/provincial/diputados_provinciales: OK
2021/municipal/concejales: OK
2025/nacional/diputados_nacionales: OK


## 4. Tabla de agrupaciones legislativas
\n\n**No se regenera ni se pisa** -- ya tiene una 4ª columna (`campo_ideologico`) clasificada a mano; este paso solo agrega, con aviso explícito, las agrupaciones nuevas que la API todavía no tenía registradas.

In [5]:
filas_agrupaciones = []
for (anio, categoria_id), info in dataframes.items():
    raw = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=f"{info['nivel']}/generales", categoria_id=categoria_id, **LA_PLATA
    )
    resultado = ResultadoElectoral.from_json(raw)
    for a in resultado.valores_totalizados_positivos:
        filas_agrupaciones.append(
            {"anio": anio, "agrupacion": a.nombre_agrupacion.upper(), "nivel": info["nivel"]}
        )

df_generado = (
    pd.DataFrame(filas_agrupaciones)
    .drop_duplicates()
    .sort_values(["anio", "nivel", "agrupacion"])
    .reset_index(drop=True)
)

# agrupaciones_legislativas.csv tiene una 4ª columna (campo_ideologico)
# clasificada a mano que la API no puede reponer -- este paso NUNCA
# sobreescribe el archivo. Solo valida si aparece alguna agrupación
# (anio, nivel, agrupacion) que todavía no esté, la informa, y la agrega con
# campo_ideologico vacío (a clasificar aparte, ver §1.1/§3 del plan de
# correcciones).
destino = REPO / "data" / "agrupaciones"
destino.mkdir(parents=True, exist_ok=True)
archivo = destino / "agrupaciones_legislativas.csv"

if archivo.exists():
    df_existente = pd.read_csv(archivo, keep_default_na=False, dtype={"anio": int})
    clave_existente = set(zip(df_existente["anio"], df_existente["nivel"], df_existente["agrupacion"]))
    es_nueva = df_generado.apply(lambda r: (r["anio"], r["nivel"], r["agrupacion"]) not in clave_existente, axis=1)
    nuevas = df_generado[es_nueva].copy()

    if nuevas.empty:
        print(f"Sin agrupaciones nuevas -- {archivo} no se modifica ({len(df_existente)} filas).")
        df_agrupaciones_legislativas = df_existente
    else:
        nuevas["campo_ideologico"] = ""
        print(f"AVISO: {len(nuevas)} agrupacion(es) nueva(s), sin clasificar todavia:")
        print(nuevas[["anio", "nivel", "agrupacion"]].to_string(index=False))
        df_agrupaciones_legislativas = (
            pd.concat([df_existente, nuevas], ignore_index=True)
            .sort_values(["anio", "nivel", "agrupacion"])
            .reset_index(drop=True)
        )
        df_agrupaciones_legislativas.to_csv(archivo, index=False)
        print(f"Se agregaron {len(nuevas)} fila(s) nueva(s) a {archivo} "
              f"(las {len(df_existente)} existentes no se tocaron).")
else:
    df_generado["campo_ideologico"] = ""
    df_generado.to_csv(archivo, index=False)
    print(f"{archivo} no existia: se creo con {len(df_generado)} filas, todas sin clasificar.")
    df_agrupaciones_legislativas = df_generado

df_agrupaciones_legislativas.head(10)

Sin agrupaciones nuevas -- /workspaces/analisis-politica-economia/data/agrupaciones/agrupaciones_legislativas.csv no se modifica (153 filas).


,anio,agrupacion,nivel,campo_ideologico
0,2013,COMPROMISO FEDERAL,municipal,4
1,2013,ESPACIO UNION PRO,municipal,5
2,2013,FRENTE CIUDAD NUEVA,municipal,1
3,2013,FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES,municipal,1
4,2013,FRENTE PARA LA VICTORIA,municipal,3
5,2013,FRENTE POPULAR DEMOCRATICO Y SOCIAL (PODEMOS),municipal,2
6,2013,FRENTE PROGRESISTA CIVICO Y SOCIAL,municipal,3
7,2013,FRENTE RENOVADOR,municipal,4
8,2013,FRENTE SOCIAL DE LA PROVINCIA DE BUENOS AIRES,municipal,3
9,2013,PARTIDO LEALTAD Y DIGNIDAD DE LA PROVINCIA DE ...,municipal,4


## 5. Estado final del caché en disco


In [6]:
for anio in ANIOS:
    for nivel in {n for n, _, _ in CARGOS}:
        carpeta = REPO / "data" / str(anio) / nivel / "generales"
        archivos = sorted(p.name for p in carpeta.iterdir()) if carpeta.exists() else []
        print(f"{anio}/{nivel}: {len(archivos)} archivos")

2013/municipal: 3 archivos
2013/provincial: 3 archivos
2013/nacional: 3 archivos
2017/municipal: 3 archivos
2017/provincial: 3 archivos
2017/nacional: 5 archivos
2021/municipal: 3 archivos
2021/provincial: 3 archivos
2021/nacional: 3 archivos
2025/municipal: 0 archivos
2025/provincial: 0 archivos
2025/nacional: 3 archivos


## 6. Ampliar etapas: PASO (§2.3 del plan de correcciones)

Mismo patrón que en `02_la_plata_cargos_ejecutivos.ipynb`: mismo cliente y
mismos `categoria_id` (`CARGOS`, arriba), cambiando `tipo_eleccion` a `1`
(PASO). El caché va a `data/<año>/<nivel>/paso/` (`categoria_nombre=f"{nivel}/paso"`).
**2025 queda afuera a propósito**: la Ley 27.781 suspendió las PASO para las
elecciones de 2025 en todo el país (verificado: la API devuelve "no
disponible" para las 5 combinaciones de `CARGOS` en 2025 con
`tipo_eleccion=1`). Los archivos de Generales que ya estaban en
`data/<año>/<nivel>/` no se tocan.

In [7]:
ANIOS_CON_PASO = [2013, 2017, 2021]  # 2025: PASO suspendidas por Ley 27.781

PASO = dict(LA_PLATA)
PASO["tipo_eleccion"] = 1  # PASO

paso_dataframes = {}
paso_faltantes = []
for anio in ANIOS_CON_PASO:
    for nivel, categoria_id, nombre in CARGOS:
        categoria_nombre = f"{nivel}/paso"
        try:
            csv_bytes = client.get_resultados_csv(
                anio_eleccion=anio, categoria_nombre=categoria_nombre, categoria_id=categoria_id, **PASO
            )
        except ResultadosNoDisponibles:
            paso_faltantes.append((anio, nombre))
            continue
        df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
        df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
        paso_dataframes[(anio, categoria_id)] = {"nivel": nivel, "nombre": nombre, "df": df}

        # JSON agregado solo se pide para los casos que el CSV confirmó disponibles
        # (igual razón que en el notebook 02: el JSON no distingue "no disponible").
        raw = client.get_resultados(
            anio_eleccion=anio, categoria_nombre=categoria_nombre, categoria_id=categoria_id, **PASO
        )
        positivos_csv = df.loc[df["votos_tipo"] == "POSITIVO", "votos_cantidad"].sum()
        positivos_json = sum(a["votos"] for a in raw["valoresTotalizadosPositivos"])
        estado = "OK" if positivos_csv == positivos_json else f"DIFERENCIA: {positivos_csv} vs {positivos_json}"
        print(f"PASO {anio}/{nivel}/{nombre} (idCargo={categoria_id}): {len(df)} filas, "
              f"{df['mesa_id'].nunique()} mesas, positivos csv vs json -> {estado}")

print("\nPASO no disponible (esperado para lo mismo que ya era 'no disponible' en Generales):")
for anio, nombre in paso_faltantes:
    print(f"  {anio}/{nombre}")

PASO 2013/nacional/diputados_nacionales (idCargo=3): 25908 filas, 1524 mesas, positivos csv vs json -> OK


PASO 2013/provincial/diputados_provinciales (idCargo=6): 24384 filas, 1524 mesas, positivos csv vs json -> OK


PASO 2013/municipal/concejales (idCargo=10): 33528 filas, 1524 mesas, positivos csv vs json -> OK


PASO 2017/nacional/senador_nacional (idCargo=2): 48227 filas, 1663 mesas, positivos csv vs json -> OK


PASO 2017/nacional/diputados_nacionales (idCargo=3): 41575 filas, 1663 mesas, positivos csv vs json -> OK


PASO 2017/provincial/diputados_provinciales (idCargo=6): 46564 filas, 1663 mesas, positivos csv vs json -> OK


PASO 2017/municipal/concejales (idCargo=10): 43428 filas, 1551 mesas, positivos csv vs json -> OK
PASO 2021/nacional/diputados_nacionales (idCargo=3): 51183 filas, 1551 mesas, positivos csv vs json -> OK


PASO 2021/provincial/diputados_provinciales (idCargo=6): 46228 filas, 1651 mesas, positivos csv vs json -> OK
PASO 2021/municipal/concejales (idCargo=10): 51181 filas, 1651 mesas, positivos csv vs json -> OK

PASO no disponible (esperado para lo mismo que ya era 'no disponible' en Generales):
  2013/senador_nacional
  2013/senadores_provinciales
  2017/senadores_provinciales
  2021/senador_nacional
  2021/senadores_provinciales


## 7. Estado final del caché para PASO

Mismo chequeo que la sección 5, ahora sobre `data/<año>/<nivel>/paso/`. La
mayoría de las carpetas debería tener exactamente 2 archivos (`.csv` +
`.json`), salvo `2017/nacional/paso`, que —igual que ya pasa con
`2017/nacional` en Generales— comparte carpeta entre dos `categoria_id`
(Senador Nacional idCargo=2, solo 2017, y Diputados Nacionales idCargo=3):
ahí se esperan 4.

No hay balotaje/segunda vuelta a nivel legislativo (Diputados/Senadores/
Concejales se reparten por sistema proporcional D'Hondt, no por mayoría con
segunda vuelta) — por eso esta sección, a diferencia del notebook 02, no
tiene una parte de balotaje.

In [8]:
ok = True
for anio, categoria_id in paso_dataframes:
    nivel = paso_dataframes[(anio, categoria_id)]["nivel"]
    carpeta = REPO / "data" / str(anio) / nivel / "paso"
    archivos = sorted(p.name for p in carpeta.iterdir())
    esperados = 4 if (anio, nivel) == (2017, "nacional") else 2
    if len(archivos) != esperados:
        ok = False
        print(f"AVISO: {carpeta} tiene {len(archivos)} archivos (esperados {esperados}): {archivos}")

print("OK: todas las carpetas de PASO tienen la cantidad de archivos esperada." if ok else "\nRevisar avisos arriba.")

OK: todas las carpetas de PASO tienen la cantidad de archivos esperada.
